# Transformers

> A good use of time, no doubt.

In [ ]:
#| default_exp transformers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn as nn, lightning.pytorch as pl, warnings
from wavefm.layers import *
from wavefm.tokenizers import *
from wavefm.patchtst import TSTBlock

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from wavefm.loss import  cosine_similarity_loss, mse_loss, huber_loss, NTXentLoss

## Contrastive Time Encoder

### Torch

In [ ]:
#| export
class PatchTFTSimpleContrastive(nn.Module):
     def __init__(self,
                   c_in,
                    patch_size,
                    patch_stride,
                    num_patches,
                    d_model,
                    n_heads,
                    d_ff,
                    num_layers,
                    augmentations=['patch_mask', 'jitter_zero_mask', 'channel_masking'],
                    mask_ratio=0.1,
                    shared_embedding=False,
                    dropout=0.0,
                    attn_dropout=0.0,
                    act='gelu',
                    pre_norm=False,
                    pe_type='tAPE',
                    qkv_bias=True,
                    init_std=0.02,
                    tokenizer_type='simple',
                    tokenizer_kwargs={},
                    pretrain_head=True,
                  ):
          super().__init__()
          self.c_in = c_in
          self.patch_size = patch_size
          self.patch_stride = patch_stride
          self.d_model = d_model
          self.num_patches = num_patches
          self.n_heads = n_heads
          self.d_ff = d_ff
          self.num_layers = num_layers
          self.act = act
          self.init_std = init_std
          self.pe_type = pe_type.lower()
          self.tokenizer_type = tokenizer_type.lower()
          self.num_patches = num_patches
          self.shared_embedding = shared_embedding
          self.pretrain_head = pretrain_head

          # Building the tokenizer

          self.patch_layer = Patch(patch_len=patch_size, stride=patch_stride)
          if self.tokenizer_type in ['simple_conv', 'simple']:
               self.tokenizer = TS_Tokenizer(
               c_in=c_in,
               patch_size=patch_size,
               d_model=d_model * c_in if not shared_embedding else d_model,
               patch_stride=patch_stride,
               shared_embedding=self.shared_embedding
               )
          elif self.tokenizer_type in ['complex_conv', 'complex']:
               self.tokenizer = TS_Tokenizer_Complex(
               c_in=c_in,
               patch_size=patch_size,
               d_model=d_model
               )
          elif self.tokenizer_type == 'linear':
               self.tokenizer = PatchEncoder(c_in=c_in, 
                                             patch_len=patch_size, 
                                             d_model=d_model,
                                             shared_embedding=self.shared_embedding
                                             )
          elif self.tokenizer_type == 'inception':
               self.tokenizer = InceptionTokenizer(c_in=c_in, 
                                                  patch_size=patch_size,
                                                  d_model=d_model * c_in if not shared_embedding else d_model,
                                                  patch_stride=patch_stride,
                                                  shared_embedding=self.shared_embedding,
                                                  **tokenizer_kwargs
                                                  )
          else:
               raise ValueError(f"Invalid tokenizer type: {tokenizer_type}. Valid options are: 'simple_conv', 'complex_conv', 'linear'")
          # Positional Encoding
          if self.pe_type == 'tape':
               self.pe = tAPE(d_model=self.d_model, seq_len=self.num_patches)
          elif self.pe_type == 'learned':
               self.pe = PositionalEncoding(num_patch=self.num_patches, d_model=self.d_model)
          elif self.pe_type == 'rotary':
               self.pe = nn.Identity()
          # residual dropout
          self.dropout = nn.Dropout(dropout)
          # time series transformer layers/Encoder
          self.layers = nn.ModuleList([TSTBlock(d_model=self.d_model, 
                                                n_heads=n_heads, 
                                                d_ff=d_ff, 
                                                attn_dropout=attn_dropout, 
                                                dropout=dropout, 
                                                bias=qkv_bias,
                                                activation=act, 
                                                pre_norm=pre_norm, 
                                                rotary_pes=self.pe_type == 'rotary') for _ in range(num_layers)])
          self.apply(self._init_weights)
          self._rescale_blocks()
          # Head
          self.mask = PatchAugmentations(augmentations=augmentations, patch_mask_ratio=mask_ratio, jitter_zero_mask_ratio=mask_ratio)
          self.projection_head = nn.Sequential(nn.Linear(self.d_model, self.d_model), nn.LayerNorm(d_model), get_activation_fn(act), nn.Linear(self.d_model, self.d_model))

     def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv1d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Conv3d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

     def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.layers):
            rescale(layer.self_attn.proj.weight.data, layer_id + 1) # rescale the attention weights
            rescale(layer.ff[3].weight.data, layer_id + 1) # rescale the feedforward weights

     def forward(self, x):
          """
          input from ds is [bs x n_vars x max_seq_len]
          z: tensor [bs x num_patch x n_vars x patch_len]
          """
          # REVIN
          bs = x.size(0)
          if x.is_nested:
               seq_len = [i.size(-1) for i in x.unbind()]
          else:
               seq_len = x.size(-1)
          x = self.patch_layer(x, constant_pad=True, constant_pad_value=0)
          
          if self.training and self.pretrain_head:
               z1 = self.mask(x) # z: [bs x num_patch x n_vars x patch_len] not implementing padding mask for masking, it shouldnt matter expects: [bs x num_patch x n_vars x patch_len]
               z2 = self.mask(x) # z: [bs x num_patch x n_vars x patch_len] not implementing padding mask for masking, it shouldnt matter expects: [bs x num_patch x n_vars x patch_len]
          else:
               z1 = z2 = x
          if (self.tokenizer_type != 'linear'):
               z1 = unpatch(z1, seq_len, remove_padding=True) # unpatch lol
               z2 = unpatch(z2, seq_len, remove_padding=True) # unpatch lol
          z1 = self.tokenizer(z1) # z: [bs x num_patch x n_vars x patch_len] pad with 0, same as input padded values
          z2 = self.tokenizer(z2) # z: [bs x num_patch x n_vars x patch_len] pad with 0, same as input padded values

          if z1.dim() == 3:
               z1 = z1.unsqueeze(2) # z: [bs x num_patch x 1 x d_model]
               z2 = z2.unsqueeze(2) # z: [bs x num_patch x 1 x d_model]
          # MASKING
          # EMBEDDING
          z1 = z1.transpose(1,2) # z: [bs x nvars x num_patch x d_model]
          z2 = z2.transpose(1,2) # z: [bs x nvars x num_patch x d_model]
          # positional encoding
          transformer_c_in = z1.size(1)
          if not z1.is_nested:
               z1 = torch.reshape(z1, (bs * transformer_c_in, self.num_patches, self.d_model)) # u: [bs * nvars x num_patch x d_model]
               z2 = torch.reshape(z2, (bs * transformer_c_in, self.num_patches, self.d_model)) # u: [bs * nvars x num_patch x d_model]
          else:
               z1_list = list(z1.unbind())
               z2_list = list(z2.unbind())
               reshaped_list = []
               for x in z_list:
                    # x has shape [c_in, var_len, d_model]
                    # Split into 7 separate tensors
                    channel_tensors = list(x.unbind(0))
                    reshaped_list.extend(channel_tensors)
               z = torch.nested.as_nested_tensor(reshaped_list, layout=torch.jagged)
          z = self.pe(z) # z: [bs * nvars x num_patch x d_model]
          # residual dropout
          z = self.dropout(z) # z: [bs * nvars x num_patch x d_model] 

          # encoder layers
          for mod in self.layers: 
               z = mod(z, mask=None) # z: [bs * n_vars x num_patch x d_model]
          if z.is_nested:
               z_list = list(z.unbind())
               restored_list = []
               for i in range(0, len(z_list), transformer_c_in):
                    group = z_list[i:i+transformer_c_in]
                    stacked = torch.stack(group, dim=0).transpose(0,1) # 1st dim is the jagged dimension
                    restored_list.append(stacked)
               z = torch.nested.as_nested_tensor(restored_list, layout=torch.jagged)
               z = z.transpose(1,2)
          else:
               z = torch.reshape(z, (-1, transformer_c_in, self.num_patches, self.d_model)) # z: [bs x nvars x num_patch x d_model]
          z = z.permute(0,1,3,2) # z: [bs x nvars x d_model x num_patch]
          if self.pretrain_head:
               Y_pred = self.head(z)
          # PRETRAIN HEAD
          if self.pretrain_head:
               return z, Y_pred, Y_true
          else:
               return z

In [ ]:
#| export
class PatchTFTSimpleTimeConstrastive(nn.Module):
     def __init__(self,
                  c_in:int, # the number of input channels
                  patch_size, # the length of the patch of time/interval or short time ft windown length (when time_domain=False)
                  patch_stride, # the length of the distance between each patch/fft
                  num_patches, # the number of patches in the sequence
                  
                  max_seq_len, # maximum sequence len
                  pos_encoding_type='learned', # options include learned or tAPE
                  use_revin=True, # if time_domain is true, whether or not to instance normalize time data
                  affine=True, # if time_domain is true, whether or not to learn revin normalization parameters 
                  mask_ratio=(0.1,0.5), # amount of signal to mask
                  augmentations=['patch_mask', 'jitter_zero_mask'], # the type of mask to use, options are patch or jitter_zero
                  n_layers:int=2, # the number of transformer encoder layers to use
                  d_model=512, # the dimension of the input to the transofmrer encoder
                  n_heads=2, # the number of heads in each layer
                  shared_embedding=False, # indicator for whether or not each channel should be projected with its own set of linear weights to the encoder dimension
                  d_ff:int=2048, # the feedforward layer size in the transformer
                  attn_dropout:float=0., # dropout in attention
                  dropout:float=0.1, # dropout for linear layers
                  act:str="gelu", # activation function
                  pre_norm:bool=False, # indicator to pre batch or layer norm 
                  pretrain=True,
                  ):
          super().__init__()
          self.c_in = c_in # original c_in without convolution
          self.pretrain = pretrain
          self.shared_embedding = shared_embedding
          self.d_model = d_model # original d_model
          self.use_revin = use_revin
          self.affine = affine
          # Instance Normalization (full sequence)
          if self.use_revin:
               self.revin = RevIN(num_features=self.c_in, affine=self.affine, dim_to_reduce=-1)
          self.num_patch = (max(max_seq_len, win_length)-win_length) // hop_length + 1
          if ((max_seq_len-win_length) % hop_length != 0):
               # add one for padding if above is true, see create_patch fxn for more details
            self.num_patch += 1
          self.patch_len = win_length
          self.patch_layer = Patch(patch_len=win_length, stride=hop_length)

          
          # Patch Embedding
          self.patch_encoder = PatchEncoder(c_in=self.c_in, patch_len=self.patch_len, d_model = self.d_model, shared_embedding=shared_embedding)
          # positional encoding
          if pos_encoding_type.lower() == 'tape':
               self.pe = tAPE(d_model=self.d_model, seq_len=self.num_patch)
          else:
               self.pe = PositionalEncoding(num_patch=self.num_patch, d_model=self.d_model)
          # residual dropout
          self.dropout = nn.Dropout(dropout)
          # time series transformer layers/Encoder
          self.layers = nn.ModuleList([TSTEncoderLayer(d_model=self.d_model, n_heads=n_heads, d_ff=d_ff,
                                                       attn_dropout=attn_dropout, dropout=dropout,
                                                       activation=act, 
                                                       pre_norm=pre_norm) for i in range(n_layers)])

          self.mask = PatchAugmentations(augmentations=augmentations, patch_mask_ratio=mask_ratio, jitter_zero_mask_ratio=mask_ratio)
          self.projection_head = nn.Sequential(nn.Linear(self.d_model, self.d_model), nn.LayerNorm(d_model), get_activation_fn(act), nn.Linear(self.d_model, self.d_model))
     def forward(self, z, channel_mask=None):
          """
          input from ds is [bs x n_vars x max_seq_len]
          z: tensor [bs x num_patch x n_vars x patch_len]
          """
          # REVIN
          if self.use_revin:
               z = self.revin(z, mode=True) # z: [bs x n_vars x max_seq_len] dont passs sequence pad mask to revin if dim=(1,) - it doesnt matter
          z = self.patch_layer(z, constant_pad=True, constant_pad_value=0) # z: [bs x num_patch x n_vars x patch_len] pad with 0, same as input padded values
          # patching for pad mask
          # MASKING
          if self.training and self.pretrain:
               z1 = self.mask(z) # z: [bs x num_patch x n_vars x patch_len] not implementing padding mask for masking, it shouldnt matter expects: [bs x num_patch x n_vars x patch_len]
               z2 = self.mask(z)
          else:
               z1 = z2 = z
          # EMBEDDING
          z1 = self.patch_encoder(z1) # z: [bs x num_patch x nvars x d_model]
          z2 = self.patch_encoder(z2) # z: [bs x num_patch x nvars x d_model]

          z1 = z1.transpose(1,2) # z: [bs x nvars x num_patch x d_model]
          z2 = z2.transpose(1,2) # z: [bs x nvars x num_patch x d_model]

          # positional encoding
          z1 = torch.reshape(z1, (-1, self.num_patch, self.d_model)) # u: [bs * nvars x num_patch x d_model]
          z2 = torch.reshape(z2, (-1, self.num_patch, self.d_model)) # u: [bs * nvars x num_patch x d_model]
          z1 = self.pe(z1) # z: [bs * nvars x num_patch x d_model]
          z2 = self.pe(z2) # z: [bs * nvars x num_patch x d_model]
          # residual dropout
          z1 = self.dropout(z1) # z: [bs * nvars x num_patch x d_model]
          z2 = self.dropout(z2) # z: [bs * nvars x num_patch x d_model]

          # encoder layers
          for mod in self.layers: 
               z1 = mod(z1, attn_mask=channel_mask) # z: [bs * n_vars x num_patch x d_model]
          for mod in self.layers: 
               z2 = mod(z2, attn_mask=channel_mask) # z: [bs * n_vars x num_patch x d_model]
          
          
          z1 = torch.reshape(z1, (-1, self.c_in, self.num_patch, self.d_model)) # z: [bs x nvars x num_patch x d_model]
          z2 = torch.reshape(z2, (-1, self.c_in, self.num_patch, self.d_model)) # z: [bs x nvars x num_patch x d_model]
          z1_proj = self.projection_head(z1)
          z2_proj = self.projection_head(z2)
          z1 = z1.permute(0,1,3,2) # z: [bs x nvars x d_model x num_patch]
          z2 = z2.permute(0,1,3,2) # z: [bs x nvars x d_model x num_patch]

          # PRETRAIN HEAD
          if self.training and self.pretrain:
               return z1_proj, z2_proj
          else:
               return z1, z1_proj, z2_proj

### Lightning

In [ ]:
#| export
class PatchTFTContrastiveLightning(pl.LightningModule):
    def __init__(self,
                 learning_rate,
                 train_size,
                 batch_size,
                 channels,
                 metrics,
                 loss_func='nt_xent',
                 optimizer_type='adamw',
                 scheduler_type='OneCycle',
                 weight_decay=0.,
                 temperature=0.2,
                 max_lr=0.01,
                 epochs=100,
                 **patchmeup_kwargs
                 ):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.max_lr = max_lr
        self.train_size = train_size
        self.batch_size = batch_size
        self.epochs = epochs
        self.metrics = metrics
        self.channels = channels
        self.loss_func = loss_func
        self.optimizer_type = optimizer_type
        self.scheduler_type = scheduler_type
        self.weight_decay = weight_decay
        self.temperature = temperature
        self.model = PatchTFTSimpleTimeConstrastive(**patchmeup_kwargs)

    def forward(self, x, channel_mask=None):
        if self.training:
            z1_proj, z2_proj = self.model(x, channel_mask=channel_mask)
            return  z1_proj, z2_proj
        else:
            z1, z1_proj, z2_proj = self.model(x, channel_mask=channel_mask)
            return z1, z1_proj, z2_proj

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, _ = batch
        f = NTXentLoss(temperature=self.temperature)
        tf_emb_1, tf_emb_2  = self.model(x)  # [bs x nvars x d_model x num_patch], this could reuturn 3 or 5 elements
        tf_emb_1_flat, tf_emb_2_flat = tf_emb_1.flatten(start_dim=-3, end_dim=-1), tf_emb_2.flatten(start_dim=-3, end_dim=-1)
        loss = f(tf_emb_1_flat, tf_emb_2_flat)
        loss = loss.to(self.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, _ = batch
        f = NTXentLoss(temperature=self.temperature)
        _, tf_emb_1, tf_emb_2 = self.model(x)  # [bs x nvars x d_model x num_patch], this could reuturn 3 or 5 elements
        tf_emb_1_flat, tf_emb_2_flat = tf_emb_1.flatten(start_dim=-3, end_dim=-1), tf_emb_2.flatten(start_dim=-3, end_dim=-1)
        loss = f(tf_emb_1_flat, tf_emb_2_flat)
        loss = loss.to(self.device)
        self.log("val_loss", loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(optimizer, max_lr=self.max_lr, total_steps=self.trainer.estimated_stepping_batches)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=self.epochs//10, T_mult=2, eta_min=1e-8) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return optimizer

## Regress Contrast

### Torch

In [ ]:
#| export
class PatchTFTRegressContrast(nn.Module):
     def __init__(self,
                  c_in:int, # the number of input channels
                  win_length, # the length of the patch of time/interval or short time ft windown length (when time_domain=False)
                  hop_length, # the length of the distance between each patch/fft
                  max_seq_len, # maximum sequence len
                  use_flash_attn=False, # indicator to use flash attention
                  use_revin=True, # if time_domain is true, whether or not to instance normalize time data
                  dim1reduce=False, # indicator to normalize by timepoint in revin
                  affine=True, # if time_domain is true, whether or not to learn revin normalization parameters 
                  mask_ratio=0.1, # amount of signal to mask
                  augmentations=['patch_mask', 'jitter_zero_mask'], # the type of mask to use, options are patch or jitter_zero
                  n_layers:int=2, # the number of transformer encoder layers to use
                  d_model=512, # the dimension of the input to the transofmrer encoder
                  n_heads=2, # the number of heads in each layer
                  shared_embedding=False, # indicator for whether or not each channel should be projected with its own set of linear weights to the encoder dimension
                  d_ff:int=2048, # the feedforward layer size in the transformer
                  norm:str='BatchNorm', # BatchNorm or LayerNorm during trianing
                  attn_dropout:float=0., # dropout in attention
                  dropout:float=0.1, # dropout for linear layers
                  act:str="gelu", # activation function
                  res_attention:bool=True, # whether to use residual attention
                  pre_norm:bool=False, # indicator to pre batch or layer norm 
                  store_attn:bool=False, # indicator to store attention
                  ):
          super().__init__()
          self.c_in = c_in # original c_in without convolution
          self.shared_embedding = shared_embedding
          self.d_model = d_model # original d_model
          self.use_revin = use_revin
          self.affine = affine
          self.use_flash_attn = use_flash_attn

          if use_flash_attn and res_attention:
               warnings.warn("Flash attention is not yet implemented for residual attention, setting res_attention=False")
               res_attention = False

          # Instance Normalization (full sequence)
          if self.use_revin:
               self.revin = RevIN(num_features=self.c_in, affine=self.affine, dim_to_reduce=1 if dim1reduce else -1)
          self.num_patch = (max(max_seq_len, win_length)-win_length) // hop_length + 1
          if ((max_seq_len-win_length) % hop_length != 0):
               # add one for padding if above is true, see create_patch fxn for more details
            self.num_patch += 1
          self.patch_len = win_length
          self.patch_layer = Patch(patch_len=win_length, stride=hop_length, max_seq_len=max_seq_len)

          
          # Patch Embedding
          self.patch_encoder = PatchEncoder(c_in=self.c_in, patch_len=self.patch_len, d_model = self.d_model, shared_embedding=shared_embedding)
          # positional encoding
          self.pe = PositionalEncoding(num_patch=self.num_patch, d_model=self.d_model)
          # residual dropout
          self.dropout = nn.Dropout(dropout)
          # time series transformer layers/Encoder
          self.layers = nn.ModuleList([TSTEncoderLayer(d_model=self.d_model, n_heads=n_heads, d_ff=d_ff, norm=norm,
                                                       attn_dropout=attn_dropout, dropout=dropout,
                                                       activation=act, res_attention=res_attention, num_patches=self.num_patch, use_flash_attn=use_flash_attn,
                                                       pre_norm=pre_norm, store_attn=store_attn) for i in range(n_layers)])
          self.res_attention = res_attention

          self.mask = PatchAugmentations(augmentations=augmentations, patch_mask_ratio=mask_ratio, jitter_zero_mask_ratio=mask_ratio)
          self.head = MaskedAutogressionFeedForward(c_in = self.c_in, patch_len = self.patch_len, d_model = self.d_model, shared_recreation=self.shared_embedding)
          self.projection_head = nn.Sequential(nn.Linear(self.d_model*self.num_patch, self.d_model), nn.BatchNorm1d(self.c_in), get_activation_fn(act), nn.Linear(self.d_model, self.d_model), nn.BatchNorm1d(self.c_in), get_activation_fn(act), nn.Linear(self.d_model, self.d_model // 2))
     def forward(self, z, sequence_padding_mask=None):
          """
          input from ds is [bs x n_vars x max_seq_len]
          z: tensor [bs x num_patch x n_vars x patch_len]
          """

          bs = z.shape[0]
          # REVIN
          if self.use_revin:
               z = self.revin(z, mode=True) # z: [bs x n_vars x max_seq_len] dont passs sequence pad mask to revin if dim=(1,) - it doesnt matter
          z = self.patch_layer(z, constant_pad=True, constant_pad_value=0) # z: [bs x num_patch x n_vars x patch_len] pad with 0, same as input padded values
          Y_true = z.clone().detach()
          # patching for pad mask
          if sequence_padding_mask is not None:
               # create a patched version of the mask for attention and loss masking
               ## pad it as well, with 1 (which is the same padding as the padded input values)
               patch_padding_mask = self.patch_layer(sequence_padding_mask, constant_pad=True, constant_pad_value=1).detach() # patch_padding_mask: [bs x num_patch x 1 x patch_len]
               key_padding_mask = torch.all(patch_padding_mask, -1).squeeze(-1) # key_padding_mask: [bs x num_patch x 1] calculate mask over patch len for each patch
               key_padding_mask = key_padding_mask.unsqueeze(1).expand(-1, self.c_in, -1) # key_padding_mask: [bs x num_patch x n_vars]
          else:
               patch_padding_mask,key_padding_mask = None, None
          # MASKING
          if self.training:
               z1 = self.mask(z) # z: [bs x num_patch x n_vars x patch_len] not implementing padding mask for masking, it shouldnt matter expects: [bs x num_patch x n_vars x patch_len]
               z2 = self.mask(z)
          else:
               z1 = z2 = z
          # EMBEDDING
          z1 = self.patch_encoder(z1) # z: [bs x num_patch x nvars x d_model]
          z2 = self.patch_encoder(z2) # z: [bs x num_patch x nvars x d_model]

          z1 = z1.transpose(1,2) # z: [bs x nvars x num_patch x d_model]
          z2 = z2.transpose(1,2) # z: [bs x nvars x num_patch x d_model]

          # positional encoding
          z1 = torch.reshape(z1, (bs*self.c_in, self.num_patch, self.d_model)) # u: [bs * nvars x num_patch x d_model]
          z2 = torch.reshape(z2, (bs*self.c_in, self.num_patch, self.d_model)) # u: [bs * nvars x num_patch x d_model]
          z1 = self.pe(z1) # z: [bs * nvars x num_patch x d_model]
          z2 = self.pe(z2) # z: [bs * nvars x num_patch x d_model]
          # residual dropout
          z1 = self.dropout(z1) # z: [bs * nvars x num_patch x d_model]
          z2 = self.dropout(z2) # z: [bs * nvars x num_patch x d_model]

          # encoder layers
          if self.res_attention:
               scores = None
               for mod in self.layers:
                    z1, scores = mod(z1, prev=scores, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model], scores: [bs * n_vars x n_heads x num_patch x num_patch]
          else:
               for mod in self.layers: 
                    z1 = mod(z1, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model]
          if self.res_attention:
               scores = None
               for mod in self.layers:
                    z2, scores = mod(z2, prev=scores, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model], scores: [bs * n_vars x n_heads x num_patch x num_patch]
          else:
               for mod in self.layers: 
                    z2 = mod(z2, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model]
          
          if key_padding_mask is not None:
               key_padding_mask = key_padding_mask.reshape(-1, self.c_in, self.num_patch)[:,0,:] # key_padding_mask: [bs x num_patch]
          
          z1 = torch.reshape(z1, (-1, self.c_in, self.num_patch, self.d_model)) # z: [bs x nvars x num_patch x d_model]
          z2 = torch.reshape(z2, (-1, self.c_in, self.num_patch, self.d_model)) # z: [bs x nvars x num_patch x d_model]
          z1_proj = z1.flatten(start_dim=2, end_dim=3) # z: [bs x nvars x d_model*num_patch]
          z2_proj = z2.flatten(start_dim=2, end_dim=3) # z: [bs x nvars x d_model*num_patch]
          z1_proj = self.projection_head(z1_proj) # z: [bs x nvars x d_model // 2]
          z2_proj = self.projection_head(z2_proj) # z: [bs x nvars x d_model // 2]
          z1 = z1.permute(0,1,3,2) # z: [bs x nvars x d_model x num_patch]
          z2 = z2.permute(0,1,3,2) # z: [bs x nvars x d_model x num_patch]
          Y_pred1 = self.head(z1)
          Y_pred2 = self.head(z2)
          
          if self.training:
               # PRETRAIN HEAD
               return z1_proj, z2_proj, Y_pred1, Y_pred2, Y_true, key_padding_mask
          else:
               return z1, z1_proj, z2_proj, Y_pred1, Y_pred2, Y_true, key_padding_mask

### Lightning

In [ ]:
#| export
class PatchTFTRegressContrastLightning(pl.LightningModule):
    def __init__(self,
                 learning_rate,
                 train_size,
                 batch_size,
                 channels,
                 metrics,
                 use_sequence_padding_mask=False,
                 loss_func='nt_xent',
                 loss_func_regression='MSE',
                 weight_decay=0.,
                 optimizer_type='Adam',
                 scheduler_type='OneCycle',
                 contrast_scaling=1.,
                 temperature=0.2,
                 max_lr=0.01,
                 epochs=100,
                 one_cycle_scheduler=True,
                 huber_delta=None, # huber loss delta
                 **patchmeup_kwargs
                 ):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.max_lr = max_lr
        self.train_size = train_size
        self.batch_size = batch_size
        self.epochs = epochs
        self.metrics = metrics
        self.one_cycle_scheduler = one_cycle_scheduler
        self.channels = channels
        self.loss_func = loss_func
        self.loss_func_regression = loss_func_regression
        self.huber_delta=huber_delta
        self.temperature = temperature
        self.contrast_scaling = contrast_scaling
        self.use_sequence_padding_mask = use_sequence_padding_mask
        self.weight_decay = weight_decay
        self.optimizer_type = optimizer_type
        self.scheduler_type = scheduler_type

        assert self.loss_func_regression.lower() in ['mae','mse','cosine','huber']
        self.model = PatchTFTRegressContrast(**patchmeup_kwargs)

    def forward(self, x, sequence_padding_mask=None):
        if self.training:
            z1_proj, z2_proj, Y_pred1, Y_pred2, Y_true, key_padding_mask  = self.model(x, sequence_padding_mask=sequence_padding_mask)  # [bs x nvars x d_model x num_patch], this could reuturn 3 or 5 elements
            return z1_proj, z2_proj, Y_pred1, Y_pred2, Y_true, key_padding_mask
        else:
            z1, z1_proj, z2_proj, Y_pred1, Y_pred2, Y_true, key_padding_mask = self.model(x, sequence_padding_mask=sequence_padding_mask)  # [bs x nvars x d_model x num_patch], this could reuturn 3 or 5 elements
            return z1, z1_proj, z2_proj, Y_pred1, Y_pred2, Y_true, key_padding_mask

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.use_sequence_padding_mask:
            x, _, sequence_padding_mask = batch
        else:
            x, _ = batch
            sequence_padding_mask = None
        nt_xent_loss = NTXentLoss(temperature=self.temperature)
        z1, z2, Y_pred1, Y_pred2, Y_true, key_padding_mask  = self.model(x, sequence_padding_mask=sequence_padding_mask)
        z1_flat, z2_flat = z1.flatten(start_dim=-2, end_dim=-1), z2.flatten(start_dim=-2, end_dim=-1)
        nt_xent_loss_val = nt_xent_loss(z1_flat, z2_flat)

        if self.loss_func_regression == 'cosine':
            reg_loss1 = cosine_similarity_loss(Y_pred1, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
            reg_loss2 = cosine_similarity_loss(Y_pred2, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
        elif self.loss_func_regression in ['MAE','mae']:
            reg_loss1 = mse_loss(Y_pred1, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
            reg_loss2 = mse_loss(Y_pred2, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
        elif self.loss_func_regression in ['Huber','huber']:
            reg_loss1 = huber_loss(Y_pred1, Y_true, None, use_mask=False, padding_mask=key_padding_mask, delta=self.huber_delta if self.huber_delta is not None else 1)
            reg_loss2 = huber_loss(Y_pred2, Y_true, None, use_mask=False, padding_mask=key_padding_mask, delta=self.huber_delta if self.huber_delta is not None else 1)
        else:
            reg_loss1 = mse_loss(Y_pred1, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
            reg_loss2 = mse_loss(Y_pred2, Y_true, None, use_mask=False, padding_mask=key_padding_mask)

        reg_loss = (reg_loss1 + reg_loss2)
        reg_loss = reg_loss.to(self.device)
        nt_xent_loss_val = nt_xent_loss_val.to(self.device)

        loss = reg_loss * (1-self.contrast_scaling) + nt_xent_loss_val*self.contrast_scaling
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_nt_xent', nt_xent_loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log(f'train_{self.loss_func_regression}', reg_loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        #self.log_dict({f'train_{metric.__name__}':metric(x_hat, y, training_mask, use_mask=self.use_mask, padding_mask=key_padding_mask) for metric in self.metrics}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        if self.use_sequence_padding_mask:
            x, _, sequence_padding_mask = batch
        else:
            x, _ = batch
            sequence_padding_mask = None
        nt_xent_loss = NTXentLoss(temperature=self.temperature)
        _, z1, z2, Y_pred1, Y_pred2, Y_true, key_padding_mask  = self.model(x, sequence_padding_mask=sequence_padding_mask)
        z1_flat, z2_flat = z1.flatten(start_dim=-2, end_dim=-1), z2.flatten(start_dim=-2, end_dim=-1)
        nt_xent_loss_val = nt_xent_loss(z1_flat, z2_flat)
        
        if self.loss_func_regression == 'cosine':
            reg_loss1 = cosine_similarity_loss(Y_pred1, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
            reg_loss2 = cosine_similarity_loss(Y_pred2, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
        elif self.loss_func_regression in ['MAE','mae']:
            reg_loss1 = mse_loss(Y_pred1, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
            reg_loss2 = mse_loss(Y_pred2, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
        elif self.loss_func_regression in ['Huber','huber']:
            reg_loss1 = huber_loss(Y_pred1, Y_true, None, use_mask=False, padding_mask=key_padding_mask, delta=self.huber_delta if self.huber_delta is not None else 1)
            reg_loss2 = huber_loss(Y_pred2, Y_true, None, use_mask=False, padding_mask=key_padding_mask, delta=self.huber_delta if self.huber_delta is not None else 1)
        else:
            reg_loss1 = mse_loss(Y_pred1, Y_true, None, use_mask=False, padding_mask=key_padding_mask)
            reg_loss2 = mse_loss(Y_pred2, Y_true, None, use_mask=False, padding_mask=key_padding_mask)

        reg_loss = (reg_loss1 + reg_loss2)
        reg_loss = reg_loss.to(self.device)
        nt_xent_loss_val = nt_xent_loss_val.to(self.device)

        loss = reg_loss *(1-self.contrast_scaling) + nt_xent_loss_val*self.contrast_scaling
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_nt_xent', nt_xent_loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log(f'val_{self.loss_func_regression}', reg_loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        #self.log_dict({f'valid_{metric.__name__}':metric(x_hat, y, training_mask, use_mask=self.use_mask, padding_mask=key_padding_mask) for metric in self.metrics}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(optimizer, max_lr=self.max_lr, total_steps=self.trainer.estimated_stepping_batches)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=self.epochs//10, T_mult=2, eta_min=1e-8) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return optimizer

## Time Frequency Contrast

### Torch

In [ ]:
#| export
class PatchTFTContrastive(nn.Module):
    def __init__(self,
                  c_in:int, # the number of input channels
                  win_length, # the length of the patch of time/interval or short time ft windown length (when time_domain=False)
                  hop_length, # the length of the distance between each patch/fft
                  max_seq_len, # maximum sequence len
                  contrast = 'both', # contrast time / frequency (both), just time, or just freq
                  use_revin=True, # if time_domain is true, whether or not to instance normalize time data
                  affine=False, # if time_domain is true, whether or not to learn revin normalization parameters 
                  dim1reduce=False, # indicator to normalize by timepoint in revin
                  mask_ratio=0.1, # amount of signal to mask
                  mask_type='jitter_zero', # the type of mask to use, options are patch or jitter_zero
                  n_layers:int=3, # the number of transformer encoder layers to use
                  d_model=128, # the dimension of the input to the transofmrer encoder
                  n_heads=16, # the number of heads in each layer
                  shared_embedding=True, # indicator for whether or not each channel should be projected with its own set of linear weights to the encoder dimension
                  d_ff:int=256, # the feedforward layer size in the transformer
                  norm:str='BatchNorm', # BatchNorm or LayerNorm during trianing
                  attn_dropout:float=0., # dropout in attention
                  dropout:float=0., # dropout for linear layers
                  act:str="gelu", # activation function
                  res_attention:bool=True, # whether to use residual attention
                  pre_norm:bool=False, # indicator to pre batch or layer norm 
                  store_attn:bool=False, # indicator to store attention
                  pre_train=True
                  ):
        super().__init__()
        self.c_in = c_in # original c_in without convolution
        self.shared_embedding = shared_embedding
        self.d_model = d_model # original d_model
        self.use_revin = use_revin
        self.affine = affine
        self.contrast = contrast
        self.pre_train = pre_train
        
        # Instance Normalization (full sequence)
        if self.use_revin:
            self.revin = RevIN(num_features=self.c_in, affine=self.affine, dim_to_reduce=1 if dim1reduce else -1)
        
        # patching
        self.num_patch = (max(max_seq_len, win_length)-win_length) // hop_length + 1
        if ((max_seq_len-win_length) % hop_length != 0):
            # add one for padding if above is true, see create_patch fxn for more details
            self.num_patch += 1
        self.patch_len = win_length
        self.patch_layer = Patch(patch_len=win_length, stride=hop_length, max_seq_len=max_seq_len)
        
        ## frequency patching
        if self.contrast in ['both', 'freq']:
            #self.channel_stft_means = channel_stft_means
            #self.channel_stft_stds = channel_stft_stds
            self.fft = FFT(dim=-1)
            #self.patch_layer_freq = STFT(n_fft=n_fft, win_length=win_length, hop_length=hop_length, stft_norm=stft_norm, decibel_scale=decibel_scale, channel_stft_means=self.channel_stft_means, channel_stft_stds=self.channel_stft_stds, pad_win_length_to_nfft=True, pad_mode='reflect', center=False, return_complex=True)
        
        # Patch Embedding
        self.identity = nn.Identity()
        if self.contrast in ['both', 'time']:
            patch_encoder_time = PatchEncoder(c_in=self.c_in, patch_len=self.patch_len, d_model = self.d_model, shared_embedding=shared_embedding)
            pe_time = PositionalEncoding(num_patch=self.num_patch, d_model=self.d_model)
            dropout_time = nn.Dropout(dropout)
            self.time_encodings = nn.Sequential(patch_encoder_time, Transpose(1,2), pe_time, dropout_time)
            self.layers_time = nn.ModuleList([TSTEncoderLayer(d_model=self.d_model, n_heads=n_heads, d_ff=d_ff, norm=norm,
                                                    attn_dropout=attn_dropout, dropout=dropout,
                                                    activation=act, res_attention=res_attention,
                                                    pre_norm=pre_norm, store_attn=store_attn) for i in range(n_layers)])
            
        if self.contrast in ['both', 'freq']:
            patch_encoder_freq = PatchEncoder(c_in=self.c_in, patch_len=self.patch_len, d_model = self.d_model, shared_embedding=shared_embedding)
            pe_freq = PositionalEncoding(num_patch=self.num_patch, d_model=self.d_model)
            dropout_freq = nn.Dropout(dropout)
            self.freq_encodings = nn.Sequential(patch_encoder_freq, Transpose(1,2), pe_freq, dropout_freq)
            self.layers_freq = nn.ModuleList([TSTEncoderLayer(d_model=self.d_model, n_heads=n_heads, d_ff=d_ff, norm=norm,
                                                    attn_dropout=attn_dropout, dropout=dropout,
                                                    activation=act, res_attention=res_attention,
                                                    pre_norm=pre_norm, store_attn=store_attn) for i in range(n_layers)])
        
        self.res_attention = res_attention
        if self.pre_train:
            self.mask = Mask(mask_ratio=mask_ratio, mask_type=mask_type, return_mask=False)
        else:
            self.mask = nn.Identity()

        if self.contrast == 'both':
            # add two final linear layers
            self.linear_time = nn.Sequential(nn.Linear(d_model, d_model), nn.BatchNorm2d(self.c_in), get_activation_fn(act), nn.Linear(d_model, d_model))
            self.linear_freq = nn.Sequential(nn.Linear(d_model, d_model), nn.BatchNorm2d(self.c_in), get_activation_fn(act), nn.Linear(d_model, d_model))
    
    def create_embeddings(self, x1, x2, key_padding_mask, mask=True, time_domain=True):
        x1, x2 = self.mask(x1) if mask else self.identity(x1), self.mask(x2) if mask else self.identity(x2) # z: [bs x num_patch x n_vars x patch_len] not implementing padding mask for masking, it shouldnt matter expects: [bs x num_patch x n_vars x patch_len]
        x1, x2 = self.time_encodings(x1) if time_domain else self.freq_encodings(x1), self.time_encodings(x2) if time_domain else self.freq_encodings(x2) # z: [bs * nvars x num_patch x d_model]
        if self.res_attention:
            scores, scores_aug = None, None
            for mod in self.layers_time if time_domain else self.layers_freq:
                x1, scores = mod(x1, prev=scores, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model], scores: [bs * n_vars x n_heads x num_patch x num_patch]
                x2, scores_aug = mod(x2, prev=scores_aug, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model], scores: [bs * n_vars x n_heads x num_patch x num_patch]
        else:
            for mod in self.layers_time if time_domain else self.layers_freq: 
                x1 = mod(x1, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model]
                x2 = mod(x2, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model]
        x1, x2 = torch.reshape(x1, (-1, self.c_in, self.num_patch, self.d_model)), torch.reshape(x2, (-1, self.c_in, self.num_patch, self.d_model)) # z: [bs x nvars x num_patch x d_model]
        return x1, x2
    
    def forward(self, z, sequence_padding_mask=None):
        """
        input from ds is [bs x n_vars x max_seq_len]
        z: tensor [bs x num_patch x n_vars x patch_len]
        """
        # REVIN
        if self.use_revin:
            z = self.revin(z, mode=True) # z: [bs x n_vars x max_seq_len] dont passs sequence pad mask to revin if dim=(1,) - it doesnt matter
        # patching
        if self.contrast in ['both','time']:
            # time domain
            z_time = self.patch_layer(z, constant_pad=True, constant_pad_value=0) # z: [bs x num_patch x n_vars x patch_len] pad with 0, same as input padded values
        if self.contrast in ['both','freq']:
            # frequency domain
            z_freq = self.fft(self.patch_layer(z, constant_pad=True, constant_pad_value=0)) # z: [bs x n_vars x n_fft // 2 + 1 x num_patch]
            #z_freq = z_freq.permute(0,3,1,2) # z: [bs x num_patch x n_vars x n_fft // 2 + 1]
        # patching for pad mask
        if sequence_padding_mask is not None:
            # create a patched version of the mask for attention and loss masking
            ## pad it as well, with 1 (which is the same padding as the padded input values)
            if self.contrast in ['both','time']:
                patch_padding_mask = self.patch_layer(sequence_padding_mask, constant_pad=True, constant_pad_value=0).detach() # patch_padding_mask: [bs x num_patch x 1 x patch_len]
                key_padding_mask = torch.all(patch_padding_mask, -1) # key_padding_mask: [bs x num_patch x 1] calculate mask over patch len for each patch
                key_padding_mask = key_padding_mask.repeat((1,1,self.c_in)) # key_padding_mask: [bs x num_patch x n_vars]
            else:
                # fft will be 0 for zero values / values with no frequency
                key_padding_mask = torch.all(z_freq, -1) # calculate mask over patch len for each patch
                key_padding_mask = (key_padding_mask[:,:,:1]==0).repeat(1,1,self.c_in) # key padding mask is true for patches with non-zero,, only need one channel dimension since channels same
        else:
            patch_padding_mask,key_padding_mask = None, None
    
        if key_padding_mask is not None:
            key_padding_mask = torch.reshape(key_padding_mask, (-1,self.num_patch)) # key_padding_mask: [bs * nvars x num_patch]
        # MASKING/augemnt
        if self.contrast == 'both':
            # time
            z_time_aug_1, z_time_aug_2 = self.create_embeddings(x1=z_time, x2=z_time, key_padding_mask=key_padding_mask, mask=True, time_domain=True) # z: [bs x nvars x num_patch x d_model]
            # frequency 
            z_freq_aug_1, z_freq_aug_2 = self.create_embeddings(x1=z_freq, x2=z_freq, key_padding_mask=key_padding_mask, mask=True, time_domain=False) # z: [bs x nvars x num_patch x d_model]
        elif self.contrast == 'time':
            # contrasting an augmented time with another augmented version
            z_time_aug_1, z_time_aug_2 = self.create_embeddings(x1=z_time, x2=z_time, key_padding_mask=key_padding_mask, mask=True, time_domain=True) # z: [bs x nvars x num_patch x d_model]
        else:
            # contrasting an augmented frequency with another augmented version
            z_freq_aug_1, z_freq_aug_2 = self.create_embeddings(x1=z_freq, x2=z_freq, key_padding_mask=key_padding_mask, mask=True, time_domain=False) # z: [bs x nvars x num_patch x d_model]
        
        if key_padding_mask is not None:
            key_padding_mask = torch.all(key_padding_mask.reshape(-1,self.num_patch if self.contrast != 'freq' else self.num_patch, self.c_in), -1) # key_padding_mask: [bs x num_patch]
        
        if self.contrast == 'both':
            z_time_p = self.linear_time(z_time_aug_1) # unaugmented versions go through linear layer  
            z_freq_p = self.linear_freq(z_freq_aug_1)
            return z_time_aug_1, z_time_p, z_time_aug_2, z_freq_aug_1, z_freq_p, z_freq_aug_2, key_padding_mask # [bs x nvars x num_patch x d_model]
        elif self.contrast == 'time':
            return z_time_aug_1, z_time_aug_2, key_padding_mask # z: [bs x nvars x num_patch x d_model]
        else:
            return z_freq_aug_1, z_freq_aug_2, key_padding_mask # [bs x nvars x num_patch x d_model]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()